In [ ]:
"""Environment configuration and project dependencies."""

import csv
import difflib
import gc
import json
import os
import re
import unicodedata
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


# Suppress TensorFlow C++ logs before importing TensorFlow-related dependencies.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Suppress non-critical warnings during notebook execution.
warnings.filterwarnings("ignore")


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

Device: cpu


In [ ]:
"""Project configuration."""

CONFIG = {
    # Dataset paths
    "train_path": (
        r"A:\Apps\Pythons\America\Project_16_With_Group"
        r"\AI_Algorithm\Datasets\train (1).csv"
    ),
    "test_path": (
        r"A:\Apps\Pythons\America\Project_16_With_Group"
        r"\AI_Algorithm\Datasets\test (1).csv"
    ),
    "lexicon_path": (
        r"A:\Apps\Pythons\America\Project_16_With_Group"
        r"\AI_Algorithm\Datasets\OA_Lexicon_eBL.csv"
    ),

    # Model directories and ensemble weights
    "model_paths": [
        r"A:\Apps\Pythons\America\Project_16_With_Group"
        r"\AI_Algorithm\archive (2)",
        r"A:\Apps\Pythons\America\Project_16_With_Group"
        r"\AI_Algorithm\archive (3)\byt5-base-akkadian_gap_setence2",
        r"A:\Apps\Pythons\America\Project_16_With_Group"
        r"\AI_Algorithm\archive (4)",
    ],
    "model_perf_weights": [0.98, 1.00, 0.40],

    # Generation parameters
    "max_length": 512,
    "max_new_tokens": 512,
    "batch_size": 8,
    "num_beams": 10,
    "length_penalty": 1.08,
    "early_stopping": True,

    # Lexicon and exact-match settings
    "oa_min_surface_freq": 3,
    "oa_min_surface_freq_ne": 2,
    "use_train_exact_match": True,

    # Output files
    "output_path": (
        r"A:\Apps\Pythons\America\Project_16_With_Group"
        r"\AI_Algorithm\output\submission.csv"
    ),
    "report_path": (
        r"A:\Apps\Pythons\America\Project_16_With_Group"
        r"\AI_Algorithm\output\reconstruction_report.csv"
    ),
}

In [ ]:
for model_path in CONFIG["model_paths"]:
    config_path = os.path.join(model_path, "config.json")

    print(f"Model path: {model_path}")
    print(f"Directory exists: {os.path.isdir(model_path)}")
    print(f"config.json exists: {os.path.isfile(config_path)}")
    print("-" * 50)

A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (2)
Exists: True
config: True
--------------------------------------------------
A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (3)\byt5-base-akkadian_gap_setence2
Exists: True
config: True
--------------------------------------------------
A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (4)
Exists: True
config: True
--------------------------------------------------


In [ ]:
# Cell 3 — Resolve valid Hugging Face model directories

def resolve_model_dir(base_path: str) -> str:
    """Return the first directory containing a Hugging Face config.json file."""
    if not os.path.isdir(base_path):
        raise FileNotFoundError(f"Model directory does not exist: {base_path}")

    config_path = os.path.join(base_path, "config.json")
    if os.path.isfile(config_path):
        return base_path

    for root, _, files in os.walk(base_path):
        if "config.json" in files:
            return root

    raise FileNotFoundError(
        f"No 'config.json' file was found under: {base_path}"
    )


resolved_models = []

for model_path in CONFIG["model_paths"]:
    try:
        resolved_path = resolve_model_dir(model_path)
        resolved_models.append(resolved_path)

        print(f"[OK] Model resolved")
        print(f"     Source:   {model_path}")
        print(f"     Resolved: {resolved_path}")

    except FileNotFoundError as error:
        print(f"[MISSING] {error}")

    print("-" * 70)


if not resolved_models:
    raise RuntimeError(
        "No valid model directories were found. "
        "Check CONFIG['model_paths'] and dataset/model attachments."
    )

CONFIG["model_paths"] = resolved_models
print(f"Resolved models: {len(resolved_models)}")

OK: A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (2) -> A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (2)
OK: A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (3)\byt5-base-akkadian_gap_setence2 -> A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (3)\byt5-base-akkadian_gap_setence2
OK: A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (4) -> A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (4)


In [ ]:
# Cell 4 — Load models and build a weighted model soup

def load_seq2seq_model(model_path: str) -> AutoModelForSeq2SeqLM:
    """Load a sequence-to-sequence model from a local checkpoint directory."""
    return AutoModelForSeq2SeqLM.from_pretrained(
        model_path,
        low_cpu_mem_usage=True,
    )


def build_weighted_state_dict(
    state_dicts: list[dict[str, torch.Tensor]],
    weights: np.ndarray,
    template_index: int,
) -> dict[str, torch.Tensor]:
    """Create a weighted average of compatible floating-point parameters."""
    final_state_dict = {}
    template_state_dict = state_dicts[template_index]

    for key, template_tensor in template_state_dict.items():
        available_tensors = [
            (weights[index], state_dict[key])
            for index, state_dict in enumerate(state_dicts)
            if key in state_dict
            and state_dict[key].shape == template_tensor.shape
        ]

        if not available_tensors:
            continue

        if not torch.is_floating_point(template_tensor):
            final_state_dict[key] = template_tensor.clone()
            continue

        total_weight = sum(weight for weight, _ in available_tensors)
        weighted_tensor = sum(
            weight * tensor.float()
            for weight, tensor in available_tensors
        )

        final_state_dict[key] = (
            weighted_tensor / total_weight
        ).to(dtype=template_tensor.dtype)

    return final_state_dict


model_paths = CONFIG["model_paths"]
model_weights = CONFIG["model_perf_weights"]

if len(model_paths) != len(model_weights):
    raise ValueError(
        "The number of model paths must match the number of model weights."
    )

if not model_paths:
    raise ValueError("No resolved model paths are available.")

print(f"Loading {len(model_paths)} model(s) on CPU...")

loaded_models = []

for model_path in model_paths:
    print(f"Loading: {model_path}")
    loaded_models.append(load_seq2seq_model(model_path))
    print("Loaded successfully.")

print("Extracting state dictionaries...")
state_dicts = [loaded_model.state_dict() for loaded_model in loaded_models]

weights = np.asarray(model_weights, dtype=np.float64)

if np.any(weights < 0) or weights.sum() <= 0:
    raise ValueError("Model weights must be non-negative with a positive total.")

normalized_weights = weights / weights.sum()
template_index = int(np.argmax(normalized_weights))

print(f"Normalized weights: {normalized_weights}")
print(f"Template model index: {template_index}")
print("Combining compatible model parameters...")

final_state_dict = build_weighted_state_dict(
    state_dicts=state_dicts,
    weights=normalized_weights,
    template_index=template_index,
)

print("Creating final ensemble model...")

model = load_seq2seq_model(model_paths[template_index])
load_result = model.load_state_dict(final_state_dict, strict=False)

if load_result.missing_keys:
    print(f"Missing keys: {len(load_result.missing_keys)}")

if load_result.unexpected_keys:
    print(f"Unexpected keys: {len(load_result.unexpected_keys)}")

model.to(DEVICE)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_paths[template_index])

del loaded_models
del state_dicts
del final_state_dict

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Model soup is ready on {DEVICE}.")

Loading models on cpu
Loading: A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (2)
Loaded successfully
Loading: A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (3)\byt5-base-akkadian_gap_setence2
Loaded successfully
Loading: A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\archive (4)
Loaded successfully
Extracting state dicts...
Normalized weights: [0.41176471 0.42016807 0.16806723]
Combining model weights...
Creating final model...
Model soup ready on cpu


In [ ]:
# Cell 5 — Normalize transliteration gaps and noise

BIG_GAP_TOKEN = "<big_gap>"
GAP_TOKEN = "<gap>"


def normalize_gaps(text: object) -> str:
    """Normalize gap markers and whitespace in transliteration text."""
    if pd.isna(text):
        return ""

    normalized_text = str(text)

    # Replace long ellipsis sequences with a large-gap token.
    normalized_text = re.sub(
        r"\.{5,}|…{2,}",
        f" {BIG_GAP_TOKEN} ",
        normalized_text,
    )

    # Replace standard gap markers with a regular-gap token.
    normalized_text = re.sub(
        r"\.{3,4}|…{1,2}",
        f" {GAP_TOKEN} ",
        normalized_text,
    )
    normalized_text = re.sub(
        r"\bxx+\b|\bx\b",
        f" {GAP_TOKEN} ",
        normalized_text,
    )

    # Collapse consecutive gap tokens.
    normalized_text = re.sub(
        rf"({re.escape(BIG_GAP_TOKEN)}\s*){{2,}}",
        f" {BIG_GAP_TOKEN} ",
        normalized_text,
    )
    normalized_text = re.sub(
        rf"({re.escape(GAP_TOKEN)}\s*){{2,}}",
        f" {GAP_TOKEN} ",
        normalized_text,
    )

    return re.sub(r"\s+", " ", normalized_text).strip()


test_df = pd.read_csv(CONFIG["test_path"])
train_df = pd.read_csv(CONFIG["train_path"])

test_df["transliteration_clean"] = test_df["transliteration"].map(normalize_gaps)
train_df["transliteration_clean"] = train_df["transliteration"].map(normalize_gaps)

test_df.head()

,id,text_id,line_start,line_end,transliteration,transliteration_clean
0,0,332fda50,1,7,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il <gap>...
1,1,332fda50,7,14,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...
2,2,332fda50,14,24,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...
3,3,332fda50,25,30,me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...,me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...


In [ ]:
# Cell 6 — Inference dataset and data loader

TRANSLATION_PREFIX = "translate Akkadian to English: "


class InferenceDataset(Dataset):
    """PyTorch dataset for sequence-to-sequence translation inference."""

    def __init__(
        self,
        dataframe: pd.DataFrame,
        tokenizer: AutoTokenizer,
        max_length: int,
    ) -> None:
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.texts = (
            TRANSLATION_PREFIX + text
            for text in dataframe["transliteration_clean"].fillna("").astype(str)
        )
        self.texts = list(self.texts)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        """Tokenize and return a single inference example."""
        encoded = self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
        }


test_dataset = InferenceDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
    max_length=CONFIG["max_length"],
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
)

print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: {CONFIG['batch_size']}")

In [ ]:
# Cell 7 — Generate predictions and collect sequence scores

raw_predictions = []
sequence_scores = []


with torch.inference_mode():
    for batch in tqdm(test_loader, desc="Generating predictions"):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

        generation_output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_beams=CONFIG["num_beams"],
            max_new_tokens=CONFIG["max_new_tokens"],
            length_penalty=CONFIG["length_penalty"],
            early_stopping=CONFIG["early_stopping"],
            output_scores=True,
            return_dict_in_generate=True,
        )

        decoded_predictions = tokenizer.batch_decode(
            generation_output.sequences,
            skip_special_tokens=True,
        )

        raw_predictions.extend(
            prediction.strip()
            for prediction in decoded_predictions
        )

        batch_sequence_scores = getattr(
            generation_output,
            "sequences_scores",
            None,
        )

        if batch_sequence_scores is None:
            sequence_scores.extend([None] * len(decoded_predictions))
        else:
            sequence_scores.extend(
                batch_sequence_scores.detach().float().cpu().tolist()
            )


if len(raw_predictions) != len(test_df):
    raise RuntimeError(
        f"Prediction count mismatch: expected {len(test_df)}, "
        f"received {len(raw_predictions)}."
    )

print(f"Generated {len(raw_predictions)} predictions.")

  0%|          | 0/1 [00:00<?, ?it/s]

Generated 4 predictions.


In [ ]:
# Cell 8 — Load OA lexicon and build a normalized token index

SUBSCRIPT_TO_ASCII = str.maketrans(
    "₀₁₂₃₄₅₆₇₈₉",
    "0123456789",
)

LEXICON_TOKEN_COLUMNS = ("form", "norm", "Alt_lex")


def normalize_lexicon_token(token: object) -> str:
    """Normalize a lexicon token for consistent dictionary lookup."""
    normalized_token = unicodedata.normalize("NFKC", str(token))
    normalized_token = normalized_token.translate(SUBSCRIPT_TO_ASCII).strip()

    normalized_token = re.sub(
        r'^[\'"“”‘’()\[\]{}<>]+|[\'"“”‘’()\[\]{}<>]+$',
        "",
        normalized_token,
    )

    return normalized_token.strip(".,;:!?").lower()


oa_lexicon = pd.read_csv(CONFIG["lexicon_path"])

required_columns = {"lexeme", "type"}
missing_columns = required_columns.difference(oa_lexicon.columns)

if missing_columns:
    raise ValueError(
        f"Missing required lexicon columns: {sorted(missing_columns)}"
    )

available_token_columns = [
    column
    for column in LEXICON_TOKEN_COLUMNS
    if column in oa_lexicon.columns
]

if not available_token_columns:
    raise ValueError(
        "None of the expected token columns are present: "
        f"{LEXICON_TOKEN_COLUMNS}"
    )

token_to_lexemes = defaultdict(set)

for row in oa_lexicon.itertuples(index=False):
    lexeme = getattr(row, "lexeme", "")
    lexeme = "" if pd.isna(lexeme) else str(lexeme).strip()

    if not lexeme:
        continue

    lexeme_type = getattr(row, "type", "")
    lexeme_type = "" if pd.isna(lexeme_type) else str(lexeme_type).strip()

    for column in available_token_columns:
        cell_value = getattr(row, column, "")

        if pd.isna(cell_value):
            continue

        for token in str(cell_value).split():
            normalized_token = normalize_lexicon_token(token)

            if normalized_token:
                token_to_lexemes[normalized_token].add(
                    (lexeme, lexeme_type)
                )

token_to_lexemes = {
    token: sorted(lexemes)
    for token, lexemes in token_to_lexemes.items()
}

print(f"OA lexicon token keys indexed: {len(token_to_lexemes):,}")

OA Lexicon token keys indexed: 54647


In [ ]:
# Cell 9 — Normalize diacritics and detect named entities

DIACRITIC_TRANSLATION_TABLE = str.maketrans(
    {
        "š": "s",
        "Š": "s",
        "ṣ": "s",
        "Ṣ": "s",
        "ṭ": "t",
        "Ṭ": "t",
        "ḫ": "h",
        "Ḫ": "h",
        "ā": "a",
        "Ā": "a",
        "ē": "e",
        "Ē": "e",
        "ī": "i",
        "Ī": "i",
        "ū": "u",
        "Ū": "u",
        "ʾ": "",
        "’": "",
        "'": "",
    }
)

DIACRITIC_CHARACTERS = frozenset(
    "šŠṣṢṭṬḫḪāēīūĀĒĪŪ"
)

NAMED_ENTITY_TYPES = frozenset(
    {"DN", "GN", "PN", "MN", "ON", "TN"}
)


def remove_disambiguation_suffix(value: object) -> str:
    """Remove trailing numeric disambiguators, such as 'ilu2' -> 'ilu'."""
    normalized_value = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"(?<=\D)\d+$", "", normalized_value)


def fold_token_for_matching(value: object) -> str:
    """Create a lowercase ASCII-like representation for fuzzy token matching."""
    folded_value = remove_disambiguation_suffix(value)
    folded_value = folded_value.translate(DIACRITIC_TRANSLATION_TABLE).lower()

    # Normalize common ASCII transliteration alternatives.
    folded_value = folded_value.replace("sh", "s").replace("kh", "h")

    return re.sub(r"[^a-z]+", "", folded_value)


def is_likely_named_entity(lexeme: object, entity_type: object) -> bool:
    """Identify likely named entities from lexicon type and orthography."""
    normalized_type = "" if pd.isna(entity_type) else str(entity_type).strip().upper()
    normalized_lexeme = "" if pd.isna(lexeme) else str(lexeme).strip()

    if normalized_type in NAMED_ENTITY_TYPES:
        return True

    has_uppercase = any(character.isupper() for character in normalized_lexeme)
    has_diacritic = any(
        character in DIACRITIC_CHARACTERS
        for character in normalized_lexeme
    )

    return has_uppercase or has_diacritic

In [ ]:
# Cell 10 — Learn canonical surface forms from training translations

TRANSLITERATION_TOKEN_PATTERN = re.compile(
    r"[A-Za-zšṣṭḫāēīūŠṢṬḪĀĒĪŪ'’-]+"
)

MIN_SURFACE_TOKEN_LENGTH = 3
MIN_FOLDED_TOKEN_LENGTH = 4

surface_form_counts = defaultdict(Counter)

target_column = (
    "translation"
    if "translation" in train_df.columns
    else train_df.columns[-1]
)

for translation in train_df[target_column].fillna("").astype(str):
    for token in TRANSLITERATION_TOKEN_PATTERN.findall(translation):
        has_name_like_case = token[0].isupper()
        has_diacritic = any(
            character in DIACRITIC_CHARACTERS
            for character in token
        )

        if len(token) < MIN_SURFACE_TOKEN_LENGTH:
            continue

        if not (has_name_like_case or has_diacritic):
            continue

        folded_token = fold_token_for_matching(token)

        if len(folded_token) >= MIN_FOLDED_TOKEN_LENGTH:
            surface_form_counts[folded_token][token] += 1


fold_to_surface = {}
fold_to_frequency = {}

for folded_token, token_counts in surface_form_counts.items():
    surface_token, frequency = token_counts.most_common(1)[0]
    fold_to_surface[folded_token] = surface_token
    fold_to_frequency[folded_token] = frequency


print(
    f"Learned {len(fold_to_surface):,} canonical surface forms "
    "from training translations."
)

Learned 1893 surface forms from train data.


In [ ]:
# Cell 11 — Normalize named entities using the OA lexicon

WORD_WITH_PUNCTUATION_PATTERN = re.compile(
    r"^(?P<prefix>\W*)(?P<token>.*?)(?P<suffix>\W*)$"
)


def extract_name_targets(
    transliteration: object,
    max_targets: int = 50,
) -> dict[str, str]:
    """Build folded-name to canonical-surface mappings from lexicon matches."""
    target_surfaces = {}
    seen_lexemes = set()

    for token in str(transliteration).split():
        normalized_token = normalize_lexicon_token(token)

        for lexeme, entity_type in token_to_lexemes.get(normalized_token, []):
            if lexeme in seen_lexemes:
                continue

            if not is_likely_named_entity(lexeme, entity_type):
                continue

            seen_lexemes.add(lexeme)

            folded_lexeme = fold_token_for_matching(
                remove_disambiguation_suffix(lexeme)
            )

            if len(folded_lexeme) < 4:
                continue

            normalized_type = str(entity_type).strip().upper()

            minimum_frequency = (
                CONFIG["oa_min_surface_freq_ne"]
                if normalized_type in NAMED_ENTITY_TYPES
                else CONFIG["oa_min_surface_freq"]
            )

            surface_form = fold_to_surface.get(folded_lexeme)
            observed_frequency = fold_to_frequency.get(folded_lexeme, 0)

            if surface_form and observed_frequency >= minimum_frequency:
                target_surfaces[folded_lexeme] = surface_form

                if len(target_surfaces) >= max_targets:
                    return target_surfaces

    return target_surfaces


def normalize_lexicon_names(
    prediction: object,
    target_surfaces: dict[str, str],
) -> str:
    """Replace eligible predicted names with learned canonical surface forms."""
    if not target_surfaces:
        return str(prediction)

    normalized_tokens = []

    for token in str(prediction).split():
        match = WORD_WITH_PUNCTUATION_PATTERN.match(token)

        if match is None:
            normalized_tokens.append(token)
            continue

        prefix = match.group("prefix")
        core_token = match.group("token")
        suffix = match.group("suffix")

        if not core_token:
            normalized_tokens.append(token)
            continue

        folded_token = fold_token_for_matching(core_token)

        should_replace = (
            len(folded_token) >= 4
            and folded_token in target_surfaces
            and core_token[:1].isupper()
        )

        if should_replace:
            normalized_tokens.append(
                f"{prefix}{target_surfaces[folded_token]}{suffix}"
            )
        else:
            normalized_tokens.append(token)

    return " ".join(normalized_tokens)

In [ ]:
# Cell 12 — Build an exact-match translation memory

def normalize_source_text(value: object) -> str:
    """Normalize whitespace for source-text lookup."""
    return re.sub(r"\s+", " ", str(value).strip())


translation_counts = defaultdict(Counter)

for source_text, target_text in zip(
    train_df["transliteration_clean"],
    train_df[target_column].fillna("").astype(str),
):
    normalized_source = normalize_source_text(source_text)
    normalized_target = target_text.strip()

    if normalized_source and normalized_target:
        translation_counts[normalized_source][normalized_target] += 1


train_exact_map = {
    source_text: target_counts.most_common(1)[0][0]
    for source_text, target_counts in translation_counts.items()
}

print(f"Translation memory entries: {len(train_exact_map):,}")

Translation memory entries: 1559


In [ ]:
# Cell 13 — Normalize punctuation and remove repeated text

DASH_TRANSLATION_TABLE = str.maketrans(
    {
        "–": "-",
        "—": "-",
        "−": "-",
    }
)

QUOTE_TRANSLATION_TABLE = str.maketrans(
    {
        "“": '"',
        "”": '"',
        "‘": "'",
        "’": "'",
    }
)


def remove_repeated_ngrams(text: str, max_ngram_size: int = 12) -> str:
    """Remove consecutively repeated words and n-grams."""
    normalized_text = re.sub(
        r"\b(\w+)(\s+\1\b)+",
        r"\1",
        text,
        flags=re.IGNORECASE,
    )

    for ngram_size in range(max_ngram_size, 1, -1):
        repeated_ngram_pattern = (
            r"\b((?:\w+[,]?\s+){"
            f"{ngram_size - 1}"
            r"}\w+)(?:\s+\1\b)+"
        )

        normalized_text = re.sub(
            repeated_ngram_pattern,
            r"\1",
            normalized_text,
            flags=re.IGNORECASE,
        )

    return normalized_text


def strip_outer_quotes(text: str) -> str:
    """Remove unmatched double quotes from the beginning or end."""
    stripped_text = text.strip()
    stripped_text = re.sub(r'^"+|"+$', "", stripped_text)

    return stripped_text.strip()


def basic_normalize(text: object) -> str:
    """Apply basic punctuation, whitespace, and repetition normalization."""
    normalized_text = str(text)
    normalized_text = normalized_text.translate(DASH_TRANSLATION_TABLE)
    normalized_text = normalized_text.translate(QUOTE_TRANSLATION_TABLE)

    normalized_text = remove_repeated_ngrams(
        normalized_text,
        max_ngram_size=12,
    )
    normalized_text = strip_outer_quotes(normalized_text)

    normalized_text = re.sub(r"[ \t]+", " ", normalized_text)
    normalized_text = re.sub(r"\s+([,.;:!?])", r"\1", normalized_text)

    return normalized_text.strip()

In [ ]:
# Cell 14 — Heuristic multi-signal confidence estimation

DEFAULT_SEQUENCE_SCORE = 0.50
MIN_CONFIDENCE = 0.05
MAX_CONFIDENCE = 0.99

TRANSLATION_MEMORY_BONUS = 0.05
SHORT_OUTPUT_PENALTY = 0.15
LONG_OUTPUT_PENALTY = 0.10


def compute_gap_penalty(
    source_text: object,
    max_penalty: float = 0.35,
) -> float:
    """Compute an uncertainty penalty based on source gap markers."""
    normalized_source = str(source_text)

    large_gap_count = normalized_source.count("<big_gap>")
    regular_gap_count = normalized_source.count("<gap>")
    source_token_count = max(len(normalized_source.split()), 1)

    gap_ratio = (
        (2 * large_gap_count) + regular_gap_count
    ) / source_token_count

    return min(gap_ratio, 1.0) * max_penalty


def compute_length_penalty(
    generated_text: object,
    min_tokens: int = 5,
    max_tokens: int = 200,
) -> float:
    """Penalize unusually short or long generated outputs."""
    token_count = len(str(generated_text).split())

    if token_count < min_tokens:
        return SHORT_OUTPUT_PENALTY

    if token_count > max_tokens:
        return LONG_OUTPUT_PENALTY

    return 0.0


def estimate_confidence(
    sequence_score: float | None,
    source_text: object,
    prediction: object,
    prediction_source: str,
) -> float:
    """Combine generation score with source and output-quality heuristics."""
    base_score = (
        DEFAULT_SEQUENCE_SCORE
        if sequence_score is None
        else float(sequence_score)
    )

    gap_penalty = compute_gap_penalty(source_text)
    length_penalty = compute_length_penalty(prediction)

    source_bonus = (
        TRANSLATION_MEMORY_BONUS
        if prediction_source == "translation_memory"
        else 0.0
    )

    confidence = base_score - gap_penalty - length_penalty + source_bonus

    return float(
        np.clip(confidence, MIN_CONFIDENCE, MAX_CONFIDENCE)
    )

In [ ]:
# Cell 15 — Heuristic morphological and grammar-pattern analysis

VERB_FINAL_PATTERNS = (
    "i-dí-in",
    "i-lá-qé",
    "i-li-kam",
    "lu-up-ta-nim",
    "i-aa-ú-mu-ni",
)

DETERMINATIVE_PATTERN = re.compile(
    r"\([a-z]+\)|\.[A-Z]+"
)

LOGOGRAM_PATTERN = re.compile(
    r"[A-Z]{2,}\.[A-Z]+|\b[A-Z]{2,}\b"
)

VERB_FINAL_PATTERN = re.compile(
    rf"({'|'.join(map(re.escape, VERB_FINAL_PATTERNS))})\W*$"
)


def analyze_grammar_pattern(
    source_text: object,
    reconstruction: object,
) -> str:
    """Generate a heuristic structural description of a transliteration."""
    normalized_source = str(source_text)
    _ = reconstruction  # Reserved for future output-based analysis.

    large_gap_count = normalized_source.count("<big_gap>")
    regular_gap_count = normalized_source.count("<gap>")
    morpheme_count = normalized_source.count("-") + 1

    has_determinative = bool(
        DETERMINATIVE_PATTERN.search(normalized_source)
    )
    has_logogram = bool(
        LOGOGRAM_PATTERN.search(normalized_source)
    )

    if large_gap_count > 0:
        completeness_note = (
            "severely fragmented; grammar is largely reconstructed"
        )
    elif regular_gap_count >= 3:
        completeness_note = (
            "moderately fragmented; grammar is partially inferred"
        )
    elif regular_gap_count > 0:
        completeness_note = (
            "minor fragmentation; grammar is mostly inferable"
        )
    else:
        completeness_note = "no explicit gap markers detected"

    structural_features = []

    if has_logogram:
        structural_features.append("contains likely Sumerian logograms")

    if has_determinative:
        structural_features.append("contains likely semantic determinatives")

    if morpheme_count > 8:
        structural_features.append("high hyphen density in transliteration")

    feature_note = (
        "; ".join(structural_features)
        if structural_features
        else "no special orthographic pattern detected"
    )

    has_verb_final_pattern = bool(
        VERB_FINAL_PATTERN.search(normalized_source)
    )

    syntax_note = (
        "matches one of the configured clause-final verbal patterns"
        if has_verb_final_pattern
        else "syntactic order was not determined by the configured patterns"
    )

    return f"{completeness_note} | {feature_note} | {syntax_note}"

In [ ]:
# Cell 16 — Rule-based semantic relationship extraction

SEMANTIC_ROLE_KEYWORDS = {
    "administrative/trade transaction record": (
        "silver",
        "shekel",
        "mina",
        "gin",
        "merchant",
        "palace",
        "investment",
    ),
    "personal/legal identification record": (
        "seal of",
        "son of",
        "daughter of",
    ),
    "personal correspondence": (
        "letter",
        "messenger",
        "word",
        "send",
        "wrote",
        "saying",
    ),
    "institutional/administrative context": (
        "colony",
        "city",
        "tablet",
    ),
}

ACTION_VERBS = (
    "gave",
    "given",
    "give",
    "sent",
    "send",
    "received",
    "receives",
    "receive",
    "took",
    "take",
    "wrote",
    "write",
    "sealed",
    "seal",
    "let",
    "said",
    "saying",
    "say",
    "came",
    "come",
    "brought",
    "bring",
    "hear",
    "heard",
)

PRONOUNS = (
    "he",
    "she",
    "they",
    "i",
    "we",
    "whoever",
    "you",
)

ACTION_PATTERN = re.compile(
    rf"\b({'|'.join(map(re.escape, ACTION_VERBS))})\b",
    flags=re.IGNORECASE,
)

PRONOUN_PATTERN = re.compile(
    rf"\b({'|'.join(map(re.escape, PRONOUNS))})\b",
    flags=re.IGNORECASE,
)


def extract_semantic_relationship(
    text: object,
    min_hits: int = 1,
    secondary_ratio: float = 0.6,
    secondary_min_hits: int = 2,
) -> dict[str, str]:
    """Extract heuristic domain, action, agent, and lexical evidence."""
    normalized_text = str(text).lower()

    category_scores = {}
    category_matches = {}

    for category, keywords in SEMANTIC_ROLE_KEYWORDS.items():
        matched_keywords = [
            keyword
            for keyword in keywords
            if keyword in normalized_text
        ]

        if len(matched_keywords) >= min_hits:
            category_scores[category] = len(matched_keywords)
            category_matches[category] = matched_keywords

    if category_scores:
        ranked_categories = sorted(
            category_scores.items(),
            key=lambda item: item[1],
            reverse=True,
        )

        primary_category, primary_score = ranked_categories[0]

        secondary_categories = [
            category
            for category, score in ranked_categories[1:]
            if score >= primary_score * secondary_ratio
            and score >= secondary_min_hits
        ]

        semantic_category = primary_category

        if secondary_categories:
            semantic_category += (
                f" (also: {', '.join(secondary_categories)})"
            )

        matched_terms = category_matches[primary_category]

    else:
        semantic_category = "context unclear, needs expert review"
        matched_terms = []

    action = "unspecified action"
    agent = "unspecified agent"

    action_match = ACTION_PATTERN.search(normalized_text)

    if action_match:
        action = action_match.group(1).lower()

        context_start = max(0, action_match.start() - 40)
        preceding_context = normalized_text[
            context_start:action_match.start()
        ]

        preceding_pronouns = list(
            PRONOUN_PATTERN.finditer(preceding_context)
        )

        if preceding_pronouns:
            agent = preceding_pronouns[-1].group(1).lower()

    if agent == "unspecified agent":
        first_pronoun = PRONOUN_PATTERN.search(normalized_text)

        if first_pronoun:
            agent = first_pronoun.group(1).lower()

    relation_summary = (
        f"agent='{agent}', action='{action}', "
        f"domain='{semantic_category}'"
    )

    evidence = (
        f"matched terms: {matched_terms}"
        if matched_terms
        else "no explicit lexical evidence"
    )

    return {
        "semantic_category": semantic_category,
        "relation_summary": relation_summary,
        "evidence": evidence,
    }

In [ ]:
# Cell 17 — Build final reconstructions and analysis report

if len(raw_predictions) != len(test_df):
    raise ValueError(
        f"Prediction count mismatch: expected {len(test_df)}, "
        f"received {len(raw_predictions)}."
    )

if len(sequence_scores) != len(test_df):
    raise ValueError(
        f"Score count mismatch: expected {len(test_df)}, "
        f"received {len(sequence_scores)}."
    )


final_records = []

for row, raw_prediction, sequence_score in zip(
    test_df.itertuples(index=False),
    raw_predictions,
    sequence_scores,
):
    source_text = row.transliteration_clean
    normalized_prediction = basic_normalize(raw_prediction)
    source_key = normalize_source_text(source_text)

    use_translation_memory = (
        CONFIG["use_train_exact_match"]
        and source_key in train_exact_map
    )

    if use_translation_memory:
        reconstruction = train_exact_map[source_key]
        reconstruction_source = "translation_memory"
    else:
        name_targets = extract_name_targets(source_text)

        reconstruction = normalize_lexicon_names(
            normalized_prediction,
            name_targets,
        )
        reconstruction_source = "model_soup"

    grammar_pattern = analyze_grammar_pattern(
        source_text,
        reconstruction,
    )
    semantic_analysis = extract_semantic_relationship(reconstruction)

    confidence = estimate_confidence(
        sequence_score=sequence_score,
        source_text=source_text,
        prediction=reconstruction,
        prediction_source=reconstruction_source,
    )

    final_records.append(
        {
            "id": row.id,
            "fragment": row.transliteration,
            "reconstruction": reconstruction,
            "confidence": round(confidence, 4),
            "grammar_pattern": grammar_pattern,
            "semantic_interpretation": (
                semantic_analysis["semantic_category"]
            ),
            "semantic_relation": (
                semantic_analysis["relation_summary"]
            ),
            "explainability_evidence": semantic_analysis["evidence"],
            "source": reconstruction_source,
        }
    )


report_df = pd.DataFrame.from_records(final_records)

print(f"Final report rows: {len(report_df):,}")
report_df.head(10)

,id,fragment,reconstruction,confidence,grammar_pattern,semantic_interpretation,semantic_relation,explainability_evidence,source
0,0,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...,From the Kanesh colony to the payment of <gap>...,0.8878,"minor fragmentation, grammar mostly inferable ...",institutional/administrative context,"agent='unspecified agent', action='come', doma...","matched terms: ['colony', 'city', 'tablet']",model_soup
1,1,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...,"<big_gap> in the tablet of the City. This day,...",0.8470,complete sentence structure | contains Sumeria...,institutional/administrative context,"agent='whoever', action='receive', domain='ins...","matched terms: ['colony', 'city', 'tablet']",model_soup
2,2,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...,In accordance with our letter you have heard o...,0.9063,complete sentence structure | contains Sumeria...,personal correspondence,"agent='he', action='given', domain='personal c...","matched terms: ['letter', 'messenger', 'send']",model_soup
3,3,me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...,I sent our certified tablet to every single or...,0.8625,complete sentence structure | contains Sumeria...,personal correspondence,"agent='i', action='sent', domain='personal cor...","matched terms: ['word', 'send', 'wrote']",model_soup


In [ ]:
# Cell 18 — Validate report coverage and output statistics

def format_coverage(mask: pd.Series, total: int) -> str:
    """Format count and percentage for a Boolean validation mask."""
    count = int(mask.sum())
    percentage = (100 * count / total) if total else 0.0

    return f"{count:,}/{total:,} ({percentage:.1f}%)"


report_size = len(report_df)

if report_size == 0:
    raise ValueError("The report dataframe is empty; no metrics can be computed.")

semantic_relations = report_df["semantic_relation"].fillna("")
evidence_text = report_df["explainability_evidence"].fillna("")

has_resolved_agent = ~semantic_relations.str.contains(
    "agent='unspecified agent'",
    regex=False,
    na=False,
)

has_resolved_action = ~semantic_relations.str.contains(
    "action='unspecified action'",
    regex=False,
    na=False,
)

has_lexical_evidence = ~evidence_text.str.contains(
    "no explicit lexical evidence",
    regex=False,
    na=False,
)

translation_memory_hits = (
    report_df["source"].eq("translation_memory")
)

confidence_values = pd.to_numeric(
    report_df["confidence"],
    errors="coerce",
)

print(
    "Agent extraction coverage:  "
    f"{format_coverage(has_resolved_agent, report_size)}"
)

print(
    "Action extraction coverage: "
    f"{format_coverage(has_resolved_action, report_size)}"
)

print(
    "Lexical evidence coverage:  "
    f"{format_coverage(has_lexical_evidence, report_size)}"
)

print(
    "Translation-memory hits:    "
    f"{format_coverage(translation_memory_hits, report_size)}"
)

print(
    "Confidence range: "
    f"{confidence_values.min():.3f} - {confidence_values.max():.3f}"
)

Agent extraction coverage:  3/4 (75.0%)
Action extraction coverage: 4/4 (100.0%)
Lexical evidence coverage:  4/4 (100.0%)
Confidence range: 0.847 - 0.906
Translation memory hits: 0/4


In [ ]:
# Cell 19 — Create output directory

OUTPUT_DIRECTORY = os.path.dirname(CONFIG["output_path"])

os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)

print(f"Output directory is ready: {OUTPUT_DIRECTORY}")

output folder ready


In [ ]:
# Cell 20 — Save submission and detailed report

SUBMISSION_COLUMNS = ["id", "reconstruction"]
PREVIEW_COLUMNS = [
    "id",
    "reconstruction",
    "confidence",
    "grammar_pattern",
    "semantic_interpretation",
    "semantic_relation",
    "explainability_evidence",
]

missing_submission_columns = set(SUBMISSION_COLUMNS).difference(report_df.columns)

if missing_submission_columns:
    raise KeyError(
        f"Missing required submission columns: "
        f"{sorted(missing_submission_columns)}"
    )

submission_df = (
    report_df[SUBMISSION_COLUMNS]
    .rename(columns={"reconstruction": "translation"})
)

submission_df.to_csv(
    CONFIG["output_path"],
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL,
)

report_df.to_csv(
    CONFIG["report_path"],
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL,
)

print(f"Submission saved: {CONFIG['output_path']}")
print(f"Detailed report saved: {CONFIG['report_path']}")

report_df[PREVIEW_COLUMNS].head(10)

Submission saved: A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\output\submission.csv
Explainable report saved: A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\output\reconstruction_report.csv


,id,reconstruction,confidence,grammar_pattern,semantic_interpretation,semantic_relation,explainability_evidence
0,0,From the Kanesh colony to the payment of <gap>...,0.8878,"minor fragmentation, grammar mostly inferable ...",institutional/administrative context,"agent='unspecified agent', action='come', doma...","matched terms: ['colony', 'city', 'tablet']"
1,1,"<big_gap> in the tablet of the City. This day,...",0.8470,complete sentence structure | contains Sumeria...,institutional/administrative context,"agent='whoever', action='receive', domain='ins...","matched terms: ['colony', 'city', 'tablet']"
2,2,In accordance with our letter you have heard o...,0.9063,complete sentence structure | contains Sumeria...,personal correspondence,"agent='he', action='given', domain='personal c...","matched terms: ['letter', 'messenger', 'send']"
3,3,I sent our certified tablet to every single or...,0.8625,complete sentence structure | contains Sumeria...,personal correspondence,"agent='i', action='sent', domain='personal cor...","matched terms: ['word', 'send', 'wrote']"


In [ ]:
# Cell 21 — Generate and save Top-K reconstruction hypotheses

TOP_K = 10
NUM_BEAMS = 10

TOPK_OUTPUT_PATH = os.path.join(
    os.path.dirname(CONFIG["output_path"]),
    "topk_hypotheses.csv",
)

if TOP_K > NUM_BEAMS:
    raise ValueError(
        "TOP_K cannot exceed NUM_BEAMS when using beam search."
    )


def generate_topk_hypotheses(
    source_text: object,
    top_k: int = TOP_K,
    num_beams: int = NUM_BEAMS,
) -> list[tuple[str, float | None]]:
    """Generate Top-K hypotheses for one normalized source sequence."""
    if top_k > num_beams:
        raise ValueError("top_k must be less than or equal to num_beams.")

    encoded_inputs = tokenizer(
        TRANSLATION_PREFIX + str(source_text),
        max_length=CONFIG["max_length"],
        truncation=True,
        return_tensors="pt",
    )

    encoded_inputs = {
        name: tensor.to(DEVICE, non_blocking=True)
        for name, tensor in encoded_inputs.items()
    }

    with torch.inference_mode():
        generation_output = model.generate(
            **encoded_inputs,
            num_beams=num_beams,
            num_return_sequences=top_k,
            max_new_tokens=CONFIG["max_new_tokens"],
            length_penalty=CONFIG["length_penalty"],
            early_stopping=CONFIG["early_stopping"],
            output_scores=True,
            return_dict_in_generate=True,
        )

    decoded_hypotheses = tokenizer.batch_decode(
        generation_output.sequences,
        skip_special_tokens=True,
    )

    sequence_scores = getattr(
        generation_output,
        "sequences_scores",
        None,
    )

    if sequence_scores is None:
        beam_scores = [None] * len(decoded_hypotheses)
    else:
        beam_scores = (
            sequence_scores.detach()
            .float()
            .cpu()
            .tolist()
        )

    hypotheses = [
        (basic_normalize(hypothesis), score)
        for hypothesis, score in zip(decoded_hypotheses, beam_scores)
    ]

    return sorted(
        hypotheses,
        key=lambda item: (
            item[1] is None,
            -(item[1] if item[1] is not None else float("-inf")),
        ),
    )


topk_records = []

for row in tqdm(
    test_df.itertuples(index=False),
    total=len(test_df),
    desc="Generating Top-K hypotheses",
):
    hypotheses = generate_topk_hypotheses(
        source_text=row.transliteration_clean,
        top_k=TOP_K,
        num_beams=NUM_BEAMS,
    )

    for rank, (hypothesis, beam_score) in enumerate(
        hypotheses,
        start=1,
    ):
        topk_records.append(
            {
                "id": row.id,
                "rank": rank,
                "hypothesis": hypothesis,
                "beam_score": (
                    round(beam_score, 4)
                    if beam_score is not None
                    else None
                ),
            }
        )


topk_df = pd.DataFrame.from_records(topk_records)

os.makedirs(os.path.dirname(TOPK_OUTPUT_PATH), exist_ok=True)

topk_df.to_csv(
    TOPK_OUTPUT_PATH,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL,
)

print(f"Top-K hypotheses saved: {TOPK_OUTPUT_PATH}")
topk_df.head(10)

Generating Top-K hypotheses:   0%|          | 0/4 [00:00<?, ?it/s]

Top-K file saved:
A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\output\topk_hypotheses.csv


,id,rank,hypothesis,raw_score
0,0,1,From the Kanesh colony to the payment of <gap>...,0.9084
1,0,2,From the Kanesh colony to the payment of <gap>...,0.9077
2,0,3,From the Kanesh colony to the payment of <gap>...,0.9063
3,0,4,From the Kanesh colony to the payment of <gap>...,0.9059
4,0,5,From the Kanesh colony to the payment of <gap>...,0.9056
5,0,6,From the Kanesh colony to the payment of <gap>...,0.9037
6,0,7,From the Kanesh colony to the payment of <gap>...,0.9028
7,0,8,From the Kanesh colony to the payment of <gap>...,0.9016
8,0,9,From the Kanesh colony to the payment of <gap>...,0.9015


In [ ]:
# Cell 21 — Visualize reconstruction confidence

CONFIDENCE_PLOT_PATH = os.path.join(
    os.path.dirname(CONFIG["output_path"]),
    "confidence_distribution.png",
)

REQUIRED_COLUMNS = {"id", "confidence"}
missing_columns = REQUIRED_COLUMNS.difference(report_df.columns)

if missing_columns:
    raise KeyError(
        f"Missing required columns for visualization: "
        f"{sorted(missing_columns)}"
    )

confidence_values = pd.to_numeric(
    report_df["confidence"],
    errors="coerce",
)

if confidence_values.isna().all():
    raise ValueError("No valid confidence values are available to plot.")

bar_colors = [
    "#01696f"
    if confidence >= 0.87
    else "#d19900"
    if confidence >= 0.80
    else "#a12c7b"
    for confidence in confidence_values.fillna(0)
]

fig = go.Figure(
    go.Bar(
        x=report_df["id"].astype(str),
        y=confidence_values,
        marker_color=bar_colors,
        text=confidence_values.round(3),
        textposition="outside",
        hovertemplate=(
            "Segment ID: %{x}<br>"
            "Confidence: %{y:.3f}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=(
        "Reconstruction Confidence by Segment"
        "<br><span style='font-size: 18px; font-weight: normal;'>"
        "Teal: >= 0.87 | Amber: 0.80–0.869 | Magenta: < 0.80"
        "</span>"
    ),
    template="plotly_white",
)

fig.update_xaxes(title_text="Segment ID")
fig.update_yaxes(
    title_text="Confidence",
    range=[0, 1],
)

fig.update_traces(cliponaxis=False)

os.makedirs(os.path.dirname(CONFIDENCE_PLOT_PATH), exist_ok=True)

fig.write_image(CONFIDENCE_PLOT_PATH, scale=2)

print(f"Confidence plot saved: {CONFIDENCE_PLOT_PATH}")

fig.show()

Confidence plot saved:
A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\output\confidence_distribution.png


In [ ]:
# Cell 22 — Generate a human-readable explainability report

from pathlib import Path


REPORT_MARKDOWN_PATH = (
    Path(CONFIG["report_path"]).parent / "explainability_report.md"
)

REQUIRED_REPORT_COLUMNS = {
    "id",
    "fragment",
    "reconstruction",
    "confidence",
    "grammar_pattern",
    "semantic_interpretation",
    "semantic_relation",
    "explainability_evidence",
    "source",
}

missing_columns = REQUIRED_REPORT_COLUMNS.difference(report_df.columns)

if missing_columns:
    raise KeyError(
        f"Missing report columns: {sorted(missing_columns)}"
    )


def escape_markdown(value: object) -> str:
    """Escape basic Markdown characters in dynamic report content."""
    text = "" if pd.isna(value) else str(value)

    for character in ("\\", "`", "*", "_", "{", "}", "[", "]", "<", ">"):
        text = text.replace(character, f"\\{character}")

    return text


report_size = len(report_df)
confidence_values = pd.to_numeric(
    report_df["confidence"],
    errors="coerce",
)

translation_memory_count = int(
    report_df["source"].eq("translation_memory").sum()
)

model_soup_count = int(
    report_df["source"].eq("model_soup").sum()
)

report_lines = [
    "# Lost Language Reconstruction Report\n\n",
    "## Overview\n\n",
    f"- Total segments reconstructed: {report_size:,}\n",
    f"- Average heuristic confidence: {confidence_values.mean():.3f}\n",
    f"- Translation-memory exact matches: {translation_memory_count:,}\n",
    f"- Model-generated reconstructions: {model_soup_count:,}\n\n",
]

for row in report_df.itertuples(index=False):
    report_lines.extend(
        [
            f"## Segment {escape_markdown(row.id)}\n\n",
            f"- **Fragment:** {escape_markdown(row.fragment)}\n",
            f"- **Reconstruction:** {escape_markdown(row.reconstruction)}\n",
            f"- **Confidence:** {float(row.confidence):.3f}\n",
            f"- **Source:** {escape_markdown(row.source)}\n",
            f"- **Grammar pattern:** {escape_markdown(row.grammar_pattern)}\n",
            (
                "- **Semantic interpretation:** "
                f"{escape_markdown(row.semantic_interpretation)}\n"
            ),
            (
                "- **Semantic relation:** "
                f"{escape_markdown(row.semantic_relation)}\n"
            ),
            (
                "- **Evidence:** "
                f"{escape_markdown(row.explainability_evidence)}\n\n"
            ),
        ]
    )

REPORT_MARKDOWN_PATH.parent.mkdir(parents=True, exist_ok=True)

REPORT_MARKDOWN_PATH.write_text(
    "".join(report_lines),
    encoding="utf-8",
)

print(f"Explainability report saved: {REPORT_MARKDOWN_PATH}")

print("\nPreview:\n")
print("".join(report_lines[:20]))

Explainability report saved:
A:\Apps\Pythons\America\Project_16_With_Group\AI_Algorithm\output\explainability_report.md

Preview:

# AI Archaeologist — Lost Language Reconstruction Report

Total segments reconstructed: 4
Average calibrated confidence: 0.876
Translation memory (exact match) hits: 0
Model-generated reconstructions: 4

## Segment 0
- **Fragment (original):** um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi„-ma mup-pu-um aa a-lim(ki) i-li-kam
- **Reconstruction:** From the Kanesh colony to the payment of <gap> datum our messengers every single day and every single day: The tablet has come to the City.
- **Confidence:** 0.888
- **Grammar pattern:** minor fragmentation, grammar mostly inferable | contains semantic determinatives; morphologically dense (agglutinative structure) | verb-final (SOV) clause
- **Semantic interpretation:** institutional/administrative context
- **Relation:** agent='unspecified agent', action='come', 

In [ ]:
# Cell 23A — Bootstrap interval for mean Top-K beam score

from pathlib import Path

CI_LEVEL = 0.95
N_BOOTSTRAP = 10_000
RANDOM_SEED = 42
MIN_BEAMS_FOR_CI = 2

TOPK_SCORE_CANDIDATES = ("beam_score", "raw_score")

DEFAULT_TOPK_PATH = (
    Path(CONFIG["output_path"]).parent / "topk_hypotheses.csv"
)


def load_topk_dataframe() -> pd.DataFrame:
    """Load Top-K hypotheses from memory or from the CSV output file."""
    if isinstance(globals().get("topk_df"), pd.DataFrame):
        print("Top-K source: existing topk_df")
        return globals()["topk_df"].copy()

    if DEFAULT_TOPK_PATH.exists():
        print(f"Top-K source: {DEFAULT_TOPK_PATH.resolve()}")
        return pd.read_csv(DEFAULT_TOPK_PATH)

    raise FileNotFoundError(
        "Top-K hypotheses were not found. Run the Top-K generation cell first."
    )


def resolve_score_column(dataframe: pd.DataFrame) -> str:
    """Return the supported score column present in the Top-K dataframe."""
    for column in TOPK_SCORE_CANDIDATES:
        if column in dataframe.columns:
            return column

    raise KeyError(
        "No supported Top-K score column was found. "
        f"Expected one of: {TOPK_SCORE_CANDIDATES}"
    )


def clean_beam_scores(values: object) -> np.ndarray:
    """Return finite numerical beam scores."""
    scores = np.asarray(values, dtype=np.float64)
    scores = scores[np.isfinite(scores)]

    if scores.size < MIN_BEAMS_FOR_CI:
        raise ValueError(
            f"At least {MIN_BEAMS_FOR_CI} valid beam scores are required."
        )

    return scores


def bootstrap_mean_interval(
    beam_scores: object,
    confidence_level: float = CI_LEVEL,
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = RANDOM_SEED,
) -> dict[str, float | int]:
    """Estimate a percentile bootstrap interval for the mean beam score."""
    if not 0 < confidence_level < 1:
        raise ValueError("confidence_level must be between 0 and 1.")

    if n_bootstrap < 1:
        raise ValueError("n_bootstrap must be at least 1.")

    scores = clean_beam_scores(beam_scores)
    random_generator = np.random.default_rng(seed)

    sample_count = scores.size
    bootstrap_indices = random_generator.integers(
        low=0,
        high=sample_count,
        size=(n_bootstrap, sample_count),
    )

    bootstrap_means = scores[bootstrap_indices].mean(axis=1)

    alpha = 1.0 - confidence_level
    ci_lower, ci_upper = np.quantile(
        bootstrap_means,
        [alpha / 2.0, 1.0 - (alpha / 2.0)],
    )

    beam_mean = scores.mean()
    beam_std = scores.std(ddof=1)
    mean_absolute_deviation = np.abs(scores - beam_mean).mean()

    return {
        "beam_mean_score": float(beam_mean),
        "beam_score_ci_lower": float(ci_lower),
        "beam_score_ci_upper": float(ci_upper),
        "beam_score_ci_width": float(ci_upper - ci_lower),
        "beam_score_std": float(beam_std),
        "beam_score_variance": float(scores.var(ddof=1)),
        "beam_score_mad": float(mean_absolute_deviation),
        "n_beams_for_ci": int(sample_count),
        "bootstrap_iterations": int(n_bootstrap),
    }


if "report_df" not in globals():
    raise NameError("report_df is not defined. Run the final pipeline first.")

if "id" not in report_df.columns:
    raise KeyError("The 'id' column is missing from report_df.")

topk_ci_df = load_topk_dataframe()
topk_score_column = resolve_score_column(topk_ci_df)

required_topk_columns = {"id", "rank", topk_score_column}
missing_columns = required_topk_columns.difference(topk_ci_df.columns)

if missing_columns:
    raise KeyError(
        f"Missing Top-K columns: {sorted(missing_columns)}"
    )

topk_ci_df = topk_ci_df.copy()
topk_ci_df[topk_score_column] = pd.to_numeric(
    topk_ci_df[topk_score_column],
    errors="coerce",
)

topk_ci_df = (
    topk_ci_df
    .dropna(subset=["id", topk_score_column])
    .sort_values(["id", "rank"])
)

if topk_ci_df.empty:
    raise ValueError("No valid numeric Top-K beam scores were found.")

beam_score_map = (
    topk_ci_df
    .groupby(topk_ci_df["id"].astype(str))[topk_score_column]
    .agg(list)
    .to_dict()
)

report_df = report_df.copy()

report_df["beam_scores"] = (
    report_df["id"]
    .astype(str)
    .map(beam_score_map)
)

missing_ids = report_df.loc[
    report_df["beam_scores"].isna(),
    "id",
].tolist()

if missing_ids:
    raise ValueError(
        "Top-K scores are missing for report IDs: "
        f"{missing_ids[:20]}"
        f"{'...' if len(missing_ids) > 20 else ''}"
    )


def calculate_segment_interval(
    row: pd.Series,
) -> pd.Series:
    """Calculate a deterministic bootstrap interval for one report row."""
    segment_seed = RANDOM_SEED + int(row.name)

    try:
        return pd.Series(
            bootstrap_mean_interval(
                beam_scores=row["beam_scores"],
                seed=segment_seed,
            )
        )
    except (TypeError, ValueError) as error:
        print(f"CI failed for id={row['id']}: {error}")

        return pd.Series(
            {
                "beam_mean_score": np.nan,
                "beam_score_ci_lower": np.nan,
                "beam_score_ci_upper": np.nan,
                "beam_score_ci_width": np.nan,
                "beam_score_std": np.nan,
                "beam_score_variance": np.nan,
                "beam_score_mad": np.nan,
                "n_beams_for_ci": np.nan,
                "bootstrap_iterations": N_BOOTSTRAP,
            }
        )


ci_results = report_df.apply(
    calculate_segment_interval,
    axis=1,
)

report_df = pd.concat(
    [
        report_df.drop(
            columns=ci_results.columns,
            errors="ignore",
        ),
        ci_results,
    ],
    axis=1,
)

report_df["beam_mean_inside_ci"] = (
    report_df["beam_score_ci_lower"].le(report_df["beam_mean_score"])
    & report_df["beam_mean_score"].le(report_df["beam_score_ci_upper"])
)

required_ci_columns = [
    "beam_mean_score",
    "beam_score_ci_lower",
    "beam_score_ci_upper",
    "beam_score_ci_width",
    "beam_score_std",
]

report_df["ci_calculation_valid"] = (
    report_df[required_ci_columns].notna().all(axis=1)
    & report_df["beam_mean_inside_ci"]
)

print("=" * 80)
print("BOOTSTRAP INTERVALS FOR MEAN BEAM SCORES")
print("=" * 80)
print(f"Confidence level: {CI_LEVEL:.0%}")
print(f"Bootstrap iterations: {N_BOOTSTRAP:,}")
print(
    "Valid interval rows: "
    f"{int(report_df['ci_calculation_valid'].sum()):,}"
    f"/{len(report_df):,}"
)
print(
    "Average interval width: "
    f"{report_df['beam_score_ci_width'].mean():.6f}"
)
print("=" * 80)

display(
    report_df[
        [
            "id",
            "confidence",
            "beam_scores",
            "beam_mean_score",
            "beam_score_ci_lower",
            "beam_score_ci_upper",
            "beam_score_ci_width",
            "beam_score_std",
            "n_beams_for_ci",
            "ci_calculation_valid",
        ]
    ].head(20)
)

Top-K source: existing topk_df
BOOTSTRAP INTERVAL FOR MEAN BEAM SCORE COMPLETE


,id,confidence,beam_confidences,beam_mean_confidence,confidence_interval_lower,confidence_interval_upper,ci_width,beam_confidence_std,beam_stability,beam_agreement,n_beams_for_ci,beam_mean_inside_ci,ci_calculation_valid
0,0,0.8878,"[0.9084, 0.9077, 0.9063, 0.9059, 0.9056, 0.903...",0.90443,0.90285,0.90605,0.00320,0.002717,99.728296,99.7650,10.0,True,True
1,1,0.8470,"[0.847, 0.846, 0.845, 0.8385, 0.8382, 0.8379, ...",0.83962,0.83681,0.84260,0.00579,0.004866,99.513429,99.6172,10.0,True,True
2,2,0.9063,"[0.9063, 0.9061, 0.9061, 0.9059, 0.9052, 0.904...",0.90336,0.90159,0.90509,0.00350,0.002981,99.701850,99.7292,10.0,True,True
3,3,0.8625,"[0.8625, 0.8621, 0.8617, 0.8611, 0.861, 0.8609...",0.85997,0.85865,0.86119,0.00254,0.002175,99.782486,99.8104,10.0,True,True



Confidence level       : 95%
Bootstrap iterations   : 10,000
Valid CI rows          : 4/4
Average Beam Count     : 10.00
Average CI Width       : 0.003757


In [42]:
# ============================================================
# Cell 23B.5
# Confidence Drift Analysis
# ============================================================

import numpy as np


def confidence_drift(row):

    model_conf = row["confidence"]

    beam_conf = row["beam_mean_confidence"]

    drift = abs(
        model_conf - beam_conf
    )

    return round(
        drift * 100,
        2
    )


report_df["confidence_drift"] = (
    report_df.apply(
        confidence_drift,
        axis=1
    )
)



def drift_label(x):

    if x <= 1:
        return "Excellent Alignment"

    elif x <= 3:
        return "Good Alignment"

    elif x <= 5:
        return "Moderate Drift"

    else:
        return "High Drift"



report_df["drift_status"] = (
    report_df["confidence_drift"]
    .apply(drift_label)
)



print("="*80)
print("Confidence Drift Analysis Complete")
print("="*80)


display(
    report_df[
        [
            "id",
            "confidence",
            "beam_mean_confidence",
            "confidence_drift",
            "drift_status"
        ]
    ]
)

Confidence Drift Analysis Complete


,id,confidence,beam_mean_confidence,confidence_drift,drift_status
0,0,0.8878,0.90443,1.66,Good Alignment
1,1,0.8470,0.83962,0.74,Excellent Alignment
2,2,0.9063,0.90336,0.29,Excellent Alignment
3,3,0.8625,0.85997,0.25,Excellent Alignment


In [36]:
# ============================================================
# Cell 23C
# AI Confidence Reliability Index (ACRI)
# ============================================================


def calculate_reliability(row):


    confidence = row["confidence"] * 100


    stability = row["beam_stability"]


    agreement = row["beam_agreement"]


    ci_width = (
        row["confidence_interval_upper"]
        -
        row["confidence_interval_lower"]
    )


    # Smaller interval = better

    ci_score = max(
        0,
        100 - (ci_width * 500)
    )


    reliability = (

        confidence * 0.40

        +

        stability * 0.25

        +

        agreement * 0.20

        +

        ci_score * 0.15

    )


    return round(
        min(reliability,100),
        2
    )



# Apply

report_df["reliability_score"] = (

    report_df.apply(

        calculate_reliability,

        axis=1

    )

)



def reliability_label(score):

    if score >= 95:
        return "🏆 Extremely Reliable"

    elif score >= 90:
        return "★★★★★ Excellent"

    elif score >= 80:
        return "★★★★ Good"

    elif score >= 70:
        return "★★★ Moderate"

    else:
        return "⚠ Low"



report_df["reliability_level"] = (

    report_df["reliability_score"]

    .apply(reliability_label)

)



print("="*80)

print(
"AI Confidence Reliability Index Complete"
)

print("="*80)


display(

    report_df[

        [

        "id",

        "confidence",

        "confidence_interval_lower",

        "confidence_interval_upper",

        "beam_stability",

        "beam_agreement",

        "reliability_score",

        "reliability_level"

        ]

    ]

)


print()

print("="*80)

print(
"Average Reliability:",
round(
report_df["reliability_score"].mean(),
2
)
)

print(
"Highest Reliability:",
report_df["reliability_score"].max()
)

print(
"Lowest Reliability:",
report_df["reliability_score"].min()
)

print("="*80)

AI Confidence Reliability Index Complete


,id,confidence,confidence_interval_lower,confidence_interval_upper,beam_stability,beam_agreement,reliability_score,reliability_level
0,0,0.8878,0.90285,0.90605,99.728296,99.7650,95.16,🏆 Extremely Reliable
1,1,0.8470,0.83681,0.84260,99.513429,99.6172,93.25,★★★★★ Excellent
2,2,0.9063,0.90159,0.90509,99.701850,99.7292,95.86,🏆 Extremely Reliable
3,3,0.8625,0.85865,0.86119,99.782486,99.8104,94.22,★★★★★ Excellent



Average Reliability: 94.62
Highest Reliability: 95.86
Lowest Reliability: 93.25


In [ ]:
# Cell 23C — Heuristic beam-interval quality score

REQUIRED_CI_COLUMNS = {
    "confidence",
    "beam_score_std",
    "beam_score_mad",
    "beam_score_ci_lower",
    "beam_score_ci_upper",
}

missing_columns = REQUIRED_CI_COLUMNS.difference(report_df.columns)

if missing_columns:
    raise KeyError(
        "Missing required columns for quality scoring: "
        f"{sorted(missing_columns)}"
    )


def calculate_interval_quality_score(row: pd.Series) -> float:
    """Calculate a descriptive quality score from pipeline diagnostics."""
    heuristic_confidence = float(row["confidence"]) * 100.0
    beam_std = float(row["beam_score_std"])
    beam_mad = float(row["beam_score_mad"])

    interval_width = (
        float(row["beam_score_ci_upper"])
        - float(row["beam_score_ci_lower"])
    )

    # These are descriptive stability measures, not calibrated probabilities.
    stability_score = np.clip((1.0 - beam_std) * 100.0, 0.0, 100.0)
    agreement_score = np.clip((1.0 - beam_mad) * 100.0, 0.0, 100.0)

    # Narrower intervals receive higher scores; width >= 0.20 receives zero.
    interval_width_score = np.clip(
        100.0 - (interval_width * 500.0),
        0.0,
        100.0,
    )

    quality_score = (
        (heuristic_confidence * 0.40)
        + (stability_score * 0.25)
        + (agreement_score * 0.25)
        + (interval_width_score * 0.10)
    )

    return round(float(quality_score), 2)


def label_interval_quality(score: float) -> str:
    """Assign a descriptive label to the heuristic score."""
    if pd.isna(score):
        return "Unavailable"

    if score >= 95:
        return "Very high heuristic consistency"

    if score >= 90:
        return "High heuristic consistency"

    if score >= 80:
        return "Moderate heuristic consistency"

    return "Needs manual review"


report_df["interval_quality_score"] = report_df.apply(
    calculate_interval_quality_score,
    axis=1,
)

report_df["interval_quality_label"] = report_df[
    "interval_quality_score"
].map(label_interval_quality)

DISPLAY_COLUMNS = [
    "id",
    "confidence",
    "beam_mean_score",
    "beam_score_ci_lower",
    "beam_score_ci_upper",
    "beam_score_ci_width",
    "beam_score_std",
    "beam_score_mad",
    "interval_quality_score",
    "interval_quality_label",
]

print("=" * 80)
print("HEURISTIC BEAM-INTERVAL QUALITY SCORING COMPLETE")
print("=" * 80)

display(report_df[DISPLAY_COLUMNS].head(30))

Confidence Interval Quality Calibration Complete


,id,confidence,confidence_interval_lower,confidence_interval_upper,beam_stability,beam_agreement,ci_quality_score,ci_quality_level
0,0,0.8878,0.90285,0.90605,99.728296,99.7650,95.23,🏆 Extremely Reliable
1,1,0.8470,0.83681,0.84260,99.513429,99.6172,93.37,★★★★★ Excellent
2,2,0.9063,0.90159,0.90509,99.701850,99.7292,95.93,🏆 Extremely Reliable
3,3,0.8625,0.85865,0.86119,99.782486,99.8104,94.27,★★★★★ Excellent


In [38]:
# ============================================================
# Cell 23D
# Confidence Interval Consistency Score
# ============================================================

import numpy as np


def calculate_consistency(row):

    confidence = row["confidence"] * 100

    beam_mean = row["beam_mean_confidence"] * 100

    stability = row["beam_stability"]

    agreement = row["beam_agreement"]

    # اختلاف Confidence و میانگین Beam
    diff = abs(confidence - beam_mean)

    consistency_from_diff = max(
        0,
        100 - diff * 4
    )

    score = (

        consistency_from_diff * 0.45 +

        stability * 0.30 +

        agreement * 0.25

    )

    return round(score, 2)


report_df["ci_consistency_score"] = (

    report_df.apply(
        calculate_consistency,
        axis=1
    )

)


def consistency_label(score):

    if score >= 95:
        return "🏆 Outstanding"

    elif score >= 90:
        return "★★★★★ Excellent"

    elif score >= 80:
        return "★★★★ Good"

    elif score >= 70:
        return "★★★ Acceptable"

    else:
        return "⚠ Inconsistent"


report_df["ci_consistency_level"] = (

    report_df["ci_consistency_score"]

    .apply(consistency_label)

)

print("="*80)
print("Confidence Interval Consistency Analysis Complete")
print("="*80)

display(

    report_df[

        [

            "id",

            "confidence",

            "beam_mean_confidence",

            "beam_stability",

            "beam_agreement",

            "ci_consistency_score",

            "ci_consistency_level"

        ]

    ]

)

print()

print("="*80)
print("Average Consistency :", round(report_df["ci_consistency_score"].mean(),2))
print("Highest Consistency :", report_df["ci_consistency_score"].max())
print("Lowest Consistency  :", report_df["ci_consistency_score"].min())
print("="*80)

Confidence Interval Consistency Analysis Complete


,id,confidence,beam_mean_confidence,beam_stability,beam_agreement,ci_consistency_score,ci_consistency_level
0,0,0.8878,0.90443,99.728296,99.7650,96.87,🏆 Outstanding
1,1,0.8470,0.83962,99.513429,99.6172,98.43,🏆 Outstanding
2,2,0.9063,0.90336,99.701850,99.7292,99.31,🏆 Outstanding
3,3,0.8625,0.85997,99.782486,99.8104,99.43,🏆 Outstanding



Average Consistency : 98.51
Highest Consistency : 99.43
Lowest Consistency  : 96.87


In [ ]:
# Cell 23D — Heuristic agreement between pipeline and beam scores

REQUIRED_AGREEMENT_COLUMNS = {
    "confidence",
    "beam_mean_score",
    "beam_score_std",
    "beam_score_mad",
}

missing_columns = REQUIRED_AGREEMENT_COLUMNS.difference(report_df.columns)

if missing_columns:
    raise KeyError(
        "Missing required columns for agreement scoring: "
        f"{sorted(missing_columns)}"
    )


def label_agreement_score(score: float) -> str:
    """Assign a descriptive label to a heuristic agreement score."""
    if pd.isna(score):
        return "Unavailable"

    if score >= 95:
        return "Very high agreement"

    if score >= 90:
        return "High agreement"

    if score >= 80:
        return "Moderate agreement"

    if score >= 70:
        return "Limited agreement"

    return "Needs manual review"


pipeline_confidence = pd.to_numeric(
    report_df["confidence"],
    errors="coerce",
) * 100.0

beam_mean_score = pd.to_numeric(
    report_df["beam_mean_score"],
    errors="coerce",
)

beam_std = pd.to_numeric(
    report_df["beam_score_std"],
    errors="coerce",
)

beam_mad = pd.to_numeric(
    report_df["beam_score_mad"],
    errors="coerce",
)

# Beam scores may be log-scores rather than probabilities; therefore, this
# comparison is descriptive only and should not be interpreted as calibration.
score_difference = (pipeline_confidence - beam_mean_score).abs()

difference_agreement = (
    100.0 - (score_difference * 4.0)
).clip(lower=0.0, upper=100.0)

beam_stability = (
    (1.0 - beam_std) * 100.0
).clip(lower=0.0, upper=100.0)

beam_agreement = (
    (1.0 - beam_mad) * 100.0
).clip(lower=0.0, upper=100.0)

report_df["beam_pipeline_agreement_score"] = (
    (difference_agreement * 0.45)
    + (beam_stability * 0.30)
    + (beam_agreement * 0.25)
).round(2)

report_df["beam_pipeline_agreement_label"] = report_df[
    "beam_pipeline_agreement_score"
].map(label_agreement_score)

DISPLAY_COLUMNS = [
    "id",
    "confidence",
    "beam_mean_score",
    "beam_score_std",
    "beam_score_mad",
    "beam_pipeline_agreement_score",
    "beam_pipeline_agreement_label",
]

print("=" * 80)
print("HEURISTIC PIPELINE–BEAM AGREEMENT ANALYSIS COMPLETE")
print("=" * 80)

display(report_df[DISPLAY_COLUMNS].head(30))

print()
print("=" * 80)
print(
    "Average agreement: "
    f"{report_df['beam_pipeline_agreement_score'].mean():.2f}"
)
print(
    "Highest agreement: "
    f"{report_df['beam_pipeline_agreement_score'].max():.2f}"
)
print(
    "Lowest agreement: "
    f"{report_df['beam_pipeline_agreement_score'].min():.2f}"
)
print("=" * 80)

In [ ]:
# Cell 23E — Final heuristic reconstruction index

FINAL_SCORE_WEIGHTS = {
    "confidence": 0.15,
    "reliability": 0.25,
    "interval_quality": 0.25,
    "beam_agreement": 0.20,
    "beam_stability": 0.075,
    "beam_consensus": 0.075,
}

REQUIRED_FINAL_COLUMNS = {
    "confidence",
    "reliability_score",
    "interval_quality_score",
    "beam_pipeline_agreement_score",
    "beam_score_std",
    "beam_score_mad",
}

missing_columns = REQUIRED_FINAL_COLUMNS.difference(report_df.columns)

if missing_columns:
    raise KeyError(
        "Missing columns for final heuristic index: "
        f"{sorted(missing_columns)}"
    )

if not np.isclose(sum(FINAL_SCORE_WEIGHTS.values()), 1.0):
    raise ValueError("Final score weights must sum to 1.0.")


def label_final_heuristic_index(score: float) -> str:
    """Return a descriptive label for a heuristic diagnostic index."""
    if pd.isna(score):
        return "Unavailable"

    if score >= 90:
        return "High heuristic consistency"

    if score >= 80:
        return "Moderate heuristic consistency"

    if score >= 70:
        return "Limited heuristic consistency"

    return "Needs manual review"


confidence_component = (
    pd.to_numeric(report_df["confidence"], errors="coerce") * 100.0
)

reliability_component = pd.to_numeric(
    report_df["reliability_score"],
    errors="coerce",
).clip(lower=0.0, upper=100.0)

interval_quality_component = pd.to_numeric(
    report_df["interval_quality_score"],
    errors="coerce",
).clip(lower=0.0, upper=100.0)

pipeline_beam_agreement_component = pd.to_numeric(
    report_df["beam_pipeline_agreement_score"],
    errors="coerce",
).clip(lower=0.0, upper=100.0)

beam_stability_component = (
    (1.0 - pd.to_numeric(
        report_df["beam_score_std"],
        errors="coerce",
    ))
    * 100.0
).clip(lower=0.0, upper=100.0)

beam_consensus_component = (
    (1.0 - pd.to_numeric(
        report_df["beam_score_mad"],
        errors="coerce",
    ))
    * 100.0
).clip(lower=0.0, upper=100.0)

report_df["final_heuristic_index"] = (
    (confidence_component * FINAL_SCORE_WEIGHTS["confidence"])
    + (reliability_component * FINAL_SCORE_WEIGHTS["reliability"])
    + (
        interval_quality_component
        * FINAL_SCORE_WEIGHTS["interval_quality"]
    )
    + (
        pipeline_beam_agreement_component
        * FINAL_SCORE_WEIGHTS["beam_agreement"]
    )
    + (
        beam_stability_component
        * FINAL_SCORE_WEIGHTS["beam_stability"]
    )
    + (
        beam_consensus_component
        * FINAL_SCORE_WEIGHTS["beam_consensus"]
    )
).round(2)

report_df["final_heuristic_label"] = report_df[
    "final_heuristic_index"
].map(label_final_heuristic_index)

DISPLAY_COLUMNS = [
    "id",
    "confidence",
    "reliability_score",
    "interval_quality_score",
    "beam_pipeline_agreement_score",
    "final_heuristic_index",
    "final_heuristic_label",
]

print("=" * 90)
print("FINAL HEURISTIC RECONSTRUCTION INDEX")
print("=" * 90)

display(report_df[DISPLAY_COLUMNS].head(30))

print()
print("=" * 90)
print(
    "Average index: "
    f"{report_df['final_heuristic_index'].mean():.2f}"
)
print(
    "Highest index: "
    f"{report_df['final_heuristic_index'].max():.2f}"
)
print(
    "Lowest index: "
    f"{report_df['final_heuristic_index'].min():.2f}"
)
print("=" * 90)

FINAL AI CONFIDENCE INDEX


,id,confidence,reliability_score,ci_quality_score,ci_consistency_score,final_confidence_index,confidence_grade
0,0,0.8878,95.16,95.23,96.87,95.25,A
1,1,0.8470,93.25,93.37,98.43,93.98,A-
2,2,0.9063,95.86,95.93,99.31,96.36,A
3,3,0.8625,94.22,94.27,99.43,94.92,A



Average Final Confidence : 95.13
Highest Final Confidence : 96.36
Lowest Final Confidence  : 93.98


In [ ]:
# Cell 23F — Heuristic reconstruction review summary

INDEX_COLUMN = "final_heuristic_index"

REQUIRED_SUMMARY_COLUMNS = {
    "id",
    "confidence",
    "reliability_score",
    "interval_quality_score",
    "beam_pipeline_agreement_score",
    INDEX_COLUMN,
}

missing_columns = REQUIRED_SUMMARY_COLUMNS.difference(report_df.columns)

if missing_columns:
    raise KeyError(
        "Missing columns for summary reporting: "
        f"{sorted(missing_columns)}"
    )


def assign_heuristic_grade(score: float) -> str:
    """Assign a descriptive grade to a heuristic index."""
    if pd.isna(score):
        return "Unavailable"

    if score >= 90:
        return "High"

    if score >= 80:
        return "Moderate"

    if score >= 70:
        return "Limited"

    return "Low"


def assign_review_priority(score: float) -> str:
    """Assign review priority; this is not an acceptance decision."""
    if pd.isna(score):
        return "Manual review required"

    if score >= 90:
        return "Low review priority"

    if score >= 80:
        return "Normal review priority"

    return "High review priority"


summary_df = report_df.copy()

summary_df["heuristic_grade"] = summary_df[INDEX_COLUMN].map(
    assign_heuristic_grade
)

summary_df["review_priority"] = summary_df[INDEX_COLUMN].map(
    assign_review_priority
)

DISPLAY_COLUMNS = [
    "id",
    "confidence",
    "reliability_score",
    "interval_quality_score",
    "beam_pipeline_agreement_score",
    INDEX_COLUMN,
    "heuristic_grade",
    "review_priority",
]

print("=" * 95)
print("HEURISTIC RECONSTRUCTION REVIEW REPORT")
print("=" * 95)

display(
    summary_df[DISPLAY_COLUMNS]
    .sort_values(INDEX_COLUMN, ascending=True)
    .head(50)
)

index_values = pd.to_numeric(
    summary_df[INDEX_COLUMN],
    errors="coerce",
)

print()
print("=" * 95)
print("Project diagnostic statistics")
print("=" * 95)
print(f"Average heuristic index: {index_values.mean():.2f}")
print(
    "Average reliability score: "
    f"{summary_df['reliability_score'].mean():.2f}"
)
print(
    "Average interval-quality score: "
    f"{summary_df['interval_quality_score'].mean():.2f}"
)
print(
    "Average pipeline–beam agreement: "
    f"{summary_df['beam_pipeline_agreement_score'].mean():.2f}"
)

print("\nReview-priority distribution:")
print(summary_df["review_priority"].value_counts(dropna=False))

print("=" * 95)

FINAL AI CONFIDENCE REPORT


,id,confidence,reliability_score,ci_quality_score,ci_consistency_score,final_confidence_index,overall_ci_grade,risk_level,decision
0,0,0.8878,95.16,95.23,96.87,95.25,A,Very Low,Accept Automatically
1,1,0.8470,93.25,93.37,98.43,93.98,A-,Low,Human Review
2,2,0.9063,95.86,95.93,99.31,96.36,A,Very Low,Accept Automatically
3,3,0.8625,94.22,94.27,99.43,94.92,A,Low,Human Review



Project Confidence Statistics
Average Final CI : 95.13
Average Reliability : 94.62
Average Quality : 94.70
Average Consistency : 98.51

Overall Project Grade : 🥇 A
